# 00 — Prepare Data

The only notebook in this project that touches the raw Dryad `.sav` files or
clones the original paper's repo (Lowet et al. 2025, `alowet/distributionalRL`).

It loads the real striatal recordings, computes a small set of derived
constants the rest of the project needs (an empirical abstraction-hierarchy
summary used only as a comparison point, and an empirically-calibrated
observation-noise covariance used to make the simulations biologically
realistic), and saves them to `artifacts/empirical_constants.npz`.

Nothing else — no raw neuron arrays, no `.sav` files — persists past this
notebook. Run this once; every other notebook only needs the small artifact
it produces.

In [6]:
import os

ARTIFACT_DIR = "artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

## Clone the paper's repo and install dependencies

Required because the `.sav` files pickle custom classes defined in the
paper's own codebase, which must be importable at unpickling time.

In [7]:
import subprocess
import sys

if not os.path.isdir("distributionalRL"):
    subprocess.run(["git", "clone", "-q", "https://github.com/alowet/distributionalRL.git"], check=True)

sys.path.insert(0, "distributionalRL")
sys.path.insert(0, "distributionalRL/utils")
sys.path.insert(0, "distributionalRL/neural_analysis")
sys.path.insert(0, "distributionalRL/behavior_analysis")

In [8]:
!pip install suite2p dPCA brainrender
!pip install mysql-connector==2.2.9
!pip install cmocean==4.0.3
!pip install bg-atlasapi==1.0.2
!pip install numpy pandas scipy scikit-learn statsmodels matplotlib seaborn plotly tqdm joblib psutil h5py scikit-image threadpoolctl loguru pingouin pymysql tensorflow suite2p brainrender bg-atlasapi cmocean vedo phy pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 3.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 36.9 MB/s eta 0:00:0000:01:00:01


## Download the required `.sav` files

Only the two ephys files this project uses (mean/variance and kurtosis
tasks); the D1/D2 imaging file isn't needed since the D1/D2 opponency
result in this project comes from the RW model, not the real imaging data.
Selective extraction avoids downloading the full ~9 GB bundle to disk twice.

In [8]:
import zipfile, requests

NP_DIR = "/content/data/neural-plots/neural-plots"
REQUIRED_SAVS = [
    f"{NP_DIR}/SameRewDist_ephys_combined_striatum_spks.sav",
    f"{NP_DIR}/SameRewVar_ephys_striatum_spks.sav",
]

TOKEN = "rve6vgZpO5UCz3QMwyK9J1zXS3AWarl9kRNFGi7amZE"
headers = {"Authorization": f"Bearer {TOKEN}"}
os.makedirs("/content/data/neural-plots", exist_ok=True)

NEURAL_PLOTS_FID = 3695811
NEURAL_PLOTS_DEST = "/content/data/neural-plots"
keep = {os.path.basename(p) for p in REQUIRED_SAVS}
BASE = "https://datadryad.org/api/v2/files/{fid}/download"

if not all(os.path.exists(p) for p in REQUIRED_SAVS):
    zip_path = "/tmp/neural_plots.zip"
    with requests.get(BASE.format(fid=NEURAL_PLOTS_FID), headers=headers, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                f.write(chunk)
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.infolist():
            if os.path.basename(member.filename) in keep:
                zf.extract(member, NEURAL_PLOTS_DEST)
    os.remove(zip_path)

for p in REQUIRED_SAVS:
    print(f"{os.path.basename(p):50s} {os.path.getsize(p)/1e6:8.0f} MB")

SameRewDist_ephys_combined_striatum_spks.sav           2249 MB
SameRewVar_ephys_striatum_spks.sav                     1095 MB


## Load the two datasets

In [9]:
import numpy as np
import pandas as pd
import joblib, pickle, warnings

warnings.filterwarnings("ignore")
np.random.seed(42)

MIN_CELLS = 20

def load_sav(path):
    try:
        return joblib.load(path, mmap_mode="r")
    except Exception:
        with open(path, "rb") as f:
            return pickle.load(f)

def get_neuron_info_df(d):
    ni = d["neuron_info"]
    return ni if isinstance(ni, pd.DataFrame) else pd.DataFrame(ni)

d_main = load_sav(REQUIRED_SAVS[0])
ni_main = get_neuron_info_df(d_main).copy()
cue_resps = d_main["cue_resps"]                 # (6, n_cells, n_trials, n_periods)
LATE = int(d_main["late_trace_ind"])
ni_main["session_id"] = ni_main["names"].astype(str) + "_" + ni_main["file_dates"].astype(str)
session_groups = ni_main.groupby("session_id").indices
msn_mask_main = (ni_main["cell_types"] == "MSN").values

d_srv = load_sav(REQUIRED_SAVS[1])
ni_srv = get_neuron_info_df(d_srv).copy()
cr_srv = d_srv["cue_resps"]
ni_srv["session_id"] = ni_srv["names"].astype(str) + "_" + ni_srv["file_dates"].astype(str)
srv_groups = ni_srv.groupby("session_id").indices
msn_mask_srv = (ni_srv["cell_types"] == "MSN").values

print(f"SameRewDist: {len(ni_main):,} neurons, {len(session_groups)} sessions, cue_resps {cue_resps.shape}")
print(f"SameRewVar:  {len(ni_srv):,} neurons, {len(srv_groups)} sessions, cr_srv {cr_srv.shape}")

SameRewDist: 13,997 neurons, 71 sessions, cue_resps (6, 13997, 90, 4)
SameRewVar:  6,816 neurons, 32 sessions, cr_srv (6, 6816, 90, 4)


## Decoding helpers

CCGP-style cross-condition decoding, inlined here so this notebook has no dependency on any other file in the project. (`01_rw_baseline.ipynb` and later notebooks import the same logic from `rw_utils.py` instead of duplicating it, since they don't need to be this self-contained.)

In [10]:
NOTHING_IDX, FIXED_IDX, VARIABLE_IDX = [0, 1], [2, 3], [4, 5]
UNIFORM_IDX, BIMODAL_IDX = [2, 3], [4, 5]

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from scipy.stats import norm

C_SVM = 5e-3
N_SAMPLE_TRIALS = 200

def make_clf():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svc", SVC(kernel="linear", C=C_SVM, class_weight="balanced", max_iter=10_000)),
    ])

def get_X(cr, cell_idx, tt, period):
    arr = cr[tt, cell_idx, :, period].T.astype(float)
    keep = ~np.isnan(arr).all(axis=1)
    arr = arr[keep]
    col_mean = np.nanmean(arr, axis=0)
    col_mean = np.where(np.isnan(col_mean), 0.0, col_mean)
    nan_mask = np.isnan(arr)
    arr[nan_mask] = np.take(col_mean, np.where(nan_mask)[1])
    return arr

def build_Xy(cr, cell_idx, neg_tts, pos_tts, period):
    Xs, ys = [], []
    for tt in neg_tts:
        X = get_X(cr, cell_idx, tt, period)
        Xs.append(X); ys.extend([0] * len(X))
    for tt in pos_tts:
        X = get_X(cr, cell_idx, tt, period)
        Xs.append(X); ys.extend([1] * len(X))
    return np.vstack(Xs), np.array(ys)

def bootstrap_X(X, n_samples=N_SAMPLE_TRIALS, rng=None):
    rng = rng or np.random.default_rng(0)
    idx = rng.choice(len(X), size=n_samples, replace=True)
    return X[idx]

def build_Xy_bootstrap(cr, cell_idx, neg_tts, pos_tts, period, n_samples=N_SAMPLE_TRIALS, rng=None):
    Xs, ys = [], []
    for tt in neg_tts:
        X = bootstrap_X(get_X(cr, cell_idx, tt, period), n_samples, rng)
        Xs.append(X); ys.extend([0] * len(X))
    for tt in pos_tts:
        X = bootstrap_X(get_X(cr, cell_idx, tt, period), n_samples, rng)
        Xs.append(X); ys.extend([1] * len(X))
    return np.vstack(Xs), np.array(ys)

def cross_identity_decode(cr, cell_idx, neg_tts, pos_tts, period, n_cv=5, seed=0,
                           n_sample_trials=N_SAMPLE_TRIALS):
    rng = np.random.default_rng(seed)
    pairings = [((neg_tts[0], pos_tts[0]), (neg_tts[1], pos_tts[1])),
                ((neg_tts[0], pos_tts[1]), (neg_tts[1], pos_tts[0]))]
    cross_accs = []
    for (negA, posA), (negB, posB) in pairings:
        ya_raw = build_Xy(cr, cell_idx, [negA], [posA], period)[1]
        yb_raw = build_Xy(cr, cell_idx, [negB], [posB], period)[1]
        if min(len(ya_raw), len(yb_raw)) < 8:
            continue
        if len(np.unique(ya_raw)) < 2 or len(np.unique(yb_raw)) < 2:
            continue
        Xa, ya = build_Xy_bootstrap(cr, cell_idx, [negA], [posA], period, n_sample_trials, rng)
        Xb, yb = build_Xy_bootstrap(cr, cell_idx, [negB], [posB], period, n_sample_trials, rng)
        for Xtr, ytr, Xte, yte in [(Xa, ya, Xb, yb), (Xb, yb, Xa, ya)]:
            clf = make_clf()
            clf.fit(Xtr, ytr)
            cross_accs.append(balanced_accuracy_score(yte, clf.predict(Xte)))
    if not cross_accs:
        return None
    cross = float(np.mean(cross_accs))

    Xa, ya = build_Xy_bootstrap(cr, cell_idx, [neg_tts[0]], [pos_tts[0]], period, n_sample_trials, rng)
    skf = StratifiedKFold(n_splits=n_cv, shuffle=True, random_state=seed)
    within = float(np.mean([
        balanced_accuracy_score(ya[te], make_clf().fit(Xa[tr], ya[tr]).predict(Xa[te]))
        for tr, te in skf.split(Xa, ya)]))

    rng_shuf = np.random.default_rng(seed + 777_000)
    Xa_s, ya_s = build_Xy_bootstrap(cr, cell_idx, [neg_tts[0]], [pos_tts[0]], period, n_sample_trials, rng_shuf)
    Xb_s, yb_s = build_Xy_bootstrap(cr, cell_idx, [neg_tts[1]], [pos_tts[1]], period, n_sample_trials, rng_shuf)
    clf_sh = make_clf()
    clf_sh.fit(Xa_s, rng_shuf.permutation(ya_s))
    shuf = float(balanced_accuracy_score(yb_s, clf_sh.predict(Xb_s)))
    return cross, within, shuf

def abstraction_index(cross, within):
    return (cross - 0.5) / (within - 0.5) if within > 0.5 else np.nan

def sdt_cross_session(cr, cell_idx, neg_tts, pos_tts, period):
    all_tts = neg_tts + pos_tts
    cond_mean = {tt: np.nanmean(cr[tt, cell_idx, :, period], axis=1) for tt in all_tts}
    valid = np.all([np.isfinite(cond_mean[tt]) for tt in all_tts], axis=0)
    if valid.sum() < 5:
        return np.nan, np.nan
    pairings = [((neg_tts[0], pos_tts[0]), (neg_tts[1], pos_tts[1])),
                ((neg_tts[0], pos_tts[1]), (neg_tts[1], pos_tts[0]))]
    cross_list, align_list = [], []
    for idA, idB in pairings:
        for (Ltr, Htr), (Lte, Hte) in [(idA, idB), (idB, idA)]:
            u_tr = (cond_mean[Htr] - cond_mean[Ltr])[valid]
            u_te = (cond_mean[Hte] - cond_mean[Lte])[valid]
            n_tr, n_te = np.linalg.norm(u_tr), np.linalg.norm(u_te)
            if n_tr < 1e-9 or n_te < 1e-9:
                continue
            align_list.append(float(u_tr @ u_te / (n_tr * n_te)))
            u_hat = u_tr / n_tr
            pL = cr[Lte, cell_idx, :, period][valid].T
            pH = cr[Hte, cell_idx, :, period][valid].T
            pL = pL[~np.isnan(pL).any(axis=1)] @ u_hat
            pH = pH[~np.isnan(pH).any(axis=1)] @ u_hat
            if len(pL) < 3 or len(pH) < 3:
                continue
            sd_pool = np.sqrt(0.5 * (pL.var(ddof=1) + pH.var(ddof=1)))
            if sd_pool < 1e-9:
                continue
            cross_list.append(float(norm.cdf(((pH.mean() - pL.mean()) / sd_pool) / 2.0)))
    if not cross_list or not align_list:
        return np.nan, np.nan
    return float(np.mean(cross_list)), float(np.mean(align_list))

print("Decoding helpers defined.")

Decoding helpers defined.


## Empirical abstraction-hierarchy summary

A lightweight, per-session CCGP/AI summary for mean, variance and kurtosis
contrasts. This is used only as a single comparison constant later ("does
the RW model's baseline hierarchy land near the real one"), not as a
result in its own right, so a simple grand mean/SEM across sessions is
sufficient here.

In [11]:
def session_hierarchy(cr, groups, msn_mask, neg_tts, pos_tts, period, min_cells=MIN_CELLS):
    rows = []
    for sess_id, idx in groups.items():
        msn_idx = idx[msn_mask[idx]]
        if len(msn_idx) < min_cells:
            continue
        res = cross_identity_decode(cr, msn_idx, neg_tts, pos_tts, period)
        if res is None:
            continue
        cross, within, _ = res
        _, cos_align = sdt_cross_session(cr, msn_idx, neg_tts, pos_tts, period)
        rows.append(dict(session_id=sess_id, cross=cross,
                          ai=abstraction_index(cross, within), cos_align=cos_align))
    return pd.DataFrame(rows)

hierarchy = {
    "mean":     session_hierarchy(cue_resps, session_groups, msn_mask_main, NOTHING_IDX,  FIXED_IDX,    LATE),
    "variance": session_hierarchy(cue_resps, session_groups, msn_mask_main, FIXED_IDX,    VARIABLE_IDX, LATE),
    "kurtosis": session_hierarchy(cr_srv,    srv_groups,     msn_mask_srv,  UNIFORM_IDX,  BIMODAL_IDX,  LATE),
}

stat_names = list(hierarchy.keys())
ai_mean       = np.array([hierarchy[s]["ai"].mean()        for s in stat_names])
ai_sem        = np.array([hierarchy[s]["ai"].sem()         for s in stat_names])
cos_align_mean = np.array([hierarchy[s]["cos_align"].mean() for s in stat_names])

for s, ai_m, ai_s, cos_m in zip(stat_names, ai_mean, ai_sem, cos_align_mean):
    n = len(hierarchy[s])
    print(f"{s:10s}  n_sessions={n:3d}  AI={ai_m:.3f}+/-{ai_s:.3f}  cos_align={cos_m:.3f}")

mean        n_sessions= 64  AI=0.623+/-0.029  cos_align=0.818
variance    n_sessions= 64  AI=0.098+/-0.024  cos_align=0.083
kurtosis    n_sessions= 31  AI=0.006+/-0.037  cos_align=0.103


## Empirically-calibrated observation-noise correlation

Real striatal populations have correlated trial-to-trial noise. This
estimates that correlation structure from held-out cells within real
sessions, tiles it to the RW population size, and returns its Cholesky
factor so the simulations can inject realistic (rather than independent)
observation noise.

In [12]:
N_CORR_UNITS = 20
N_UNITS_RW = 50
FINAL_OBS_NOISE = 5.0   # observation-noise scale the calibration is expressed in

def estimate_session_corr(cr, cell_idx, tt, period, n_sub=N_CORR_UNITS, seed=0):
    rng = np.random.default_rng(seed)
    X = cr[tt, cell_idx, :, period].T.astype(float)
    X = X[~np.isnan(X).any(axis=1)]
    if X.shape[0] < 10 or X.shape[1] < n_sub:
        return None
    idx = rng.choice(X.shape[1], n_sub, replace=False)
    resid = X[:, idx] - X[:, idx].mean(axis=0)
    return np.corrcoef(resid.T)

corr_matrices = []
for sess_id, idx in session_groups.items():
    msn_idx = idx[msn_mask_main[idx]]
    if len(msn_idx) < N_CORR_UNITS:
        continue
    for tt in range(6):
        C = estimate_session_corr(cue_resps, msn_idx, tt, LATE, seed=int(tt))
        if C is not None:
            corr_matrices.append(C)

C_pooled = np.nanmean(corr_matrices, axis=0)
offdiag = ~np.eye(N_CORR_UNITS, dtype=bool)
print(f"Pooled off-diagonal noise correlation: {C_pooled[offdiag].mean():.4f} "
      f"(from {len(corr_matrices)} session x trial-type pairs)")

reps = int(np.ceil(N_UNITS_RW / N_CORR_UNITS))
C_full = np.tile(C_pooled, (reps, reps))[:N_UNITS_RW, :N_UNITS_RW]
C_full = C_full + np.eye(N_UNITS_RW) * 1e-3
eigvals, eigvecs = np.linalg.eigh(C_full)
eigvals = np.clip(eigvals, 1e-6, None)
C_full_pd = eigvecs @ np.diag(eigvals) @ eigvecs.T
L_pooled = np.linalg.cholesky(C_full_pd * FINAL_OBS_NOISE ** 2)
print(f"L_pooled computed: shape {L_pooled.shape}, calibrated for obs_noise_sd={FINAL_OBS_NOISE}")

Pooled off-diagonal noise correlation: 0.0366 (from 384 session x trial-type pairs)
L_pooled computed: shape (50, 50), calibrated for obs_noise_sd=5.0


## Trial/cell-count calibration

So the simulated regime (trials per odor, cells per population) can be
grounded in the real per-session numbers rather than picked arbitrarily.

In [13]:
def median_valid_trials(cr, groups, tt, period):
    counts = []
    for sess_id, idx in groups.items():
        if len(idx) == 0:
            continue
        vals = cr[tt, idx[0], :, period]
        counts.append(int(np.sum(~np.isnan(vals))))
    return float(np.median(counts))

median_trials_per_condition = float(np.median(
    [median_valid_trials(cue_resps, session_groups, tt, LATE) for tt in range(6)]))
median_n_msn_per_session = float(np.median(
    [msn_mask_main[idx].sum() for idx in session_groups.values()]))

print(f"Median valid trials per condition per session: {median_trials_per_condition:.0f}")
print(f"Median MSNs per session: {median_n_msn_per_session:.0f}")

Median valid trials per condition per session: 35
Median MSNs per session: 88


## Save the artifact bundle

In [14]:
out_path = os.path.join(ARTIFACT_DIR, "empirical_constants.npz")
np.savez(
    out_path,
    stat_names=np.array(stat_names),
    ai_mean=ai_mean,
    ai_sem=ai_sem,
    cos_align_mean=cos_align_mean,
    L_pooled=L_pooled,
    n_units_rw=N_UNITS_RW,
    obs_noise_sd=FINAL_OBS_NOISE,
    median_trials_per_condition=median_trials_per_condition,
    median_n_msn_per_session=median_n_msn_per_session,
)
print(f"Saved {out_path} ({os.path.getsize(out_path)/1e3:.1f} KB)")
print(f"Contents: {list(np.load(out_path).keys())}")

Saved artifacts/empirical_constants.npz (22.5 KB)
Contents: ['stat_names', 'ai_mean', 'ai_sem', 'cos_align_mean', 'L_pooled', 'n_units_rw', 'obs_noise_sd', 'median_trials_per_condition', 'median_n_msn_per_session']


## Download the artifact

Run this notebook only on Colab. Everything else (`01_rw_baseline.ipynb` onward) runs locally against the downloaded `empirical_constants.npz` placed in this project's `artifacts/` folder.

In [ ]:
import shutil
shutil.make_archive(ARTIFACT_DIR, 'zip', ARTIFACT_DIR)

from google.colab import files
files.download(f"{ARTIFACT_DIR}.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>